# Category 5 - Memory Architecture Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares memory architectures for Ally Vision v2. The project needs persistent multilingual memory with low ops overhead, startup preload, and semantic recall — not just a generic agent framework checklist.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Memory store code": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py",
    "Memory manager code": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_manager.py",
    "Realtime route": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py",
    "Python sqlite3 docs": "https://docs.python.org/3/library/sqlite3.html"
  },
  "comparison_rows": [
    {
      "Metric": "Cross-session persistence",
      "SQLite + embeddings": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "No [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Yes [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Yes [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "Yes [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "Yes [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "Yes [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Yes [source: https://qdrant.tech/documentation/]",
      "mem0": "Yes [source: https://docs.mem0.ai/]",
      "Zep": "Yes [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "Yes [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Extra service required",
      "SQLite + embeddings": "No extra service [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "No [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Usually yes or local daemon [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Yes [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "Yes [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "Yes [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "Usually yes [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Yes [source: https://qdrant.tech/documentation/]",
      "mem0": "Yes [source: https://docs.mem0.ai/]",
      "Zep": "Yes [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "Depends on storage backend [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Write latency (qualitative)",
      "SQLite + embeddings": "Low, local file write [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "N/A (not publicly available) [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "N/A (not publicly available) [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Network dependent [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "Network dependent [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "Low [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Low to medium [source: https://qdrant.tech/documentation/]",
      "mem0": "N/A (not publicly available) [source: https://docs.mem0.ai/]",
      "Zep": "N/A (not publicly available) [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Recall latency (qualitative)",
      "SQLite + embeddings": "Low for local-scale memory set [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "No retrieval phase [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Low to medium [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Network dependent [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "Network dependent [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "N/A (not publicly available) [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Low to medium [source: https://qdrant.tech/documentation/]",
      "mem0": "N/A (not publicly available) [source: https://docs.mem0.ai/]",
      "Zep": "N/A (not publicly available) [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Hybrid search support",
      "SQLite + embeddings": "Not built-in; project uses semantic retrieval [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "N/A (not publicly available) [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Possible [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Yes with vendor features [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "Yes [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "N/A (not publicly available) [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Yes [source: https://qdrant.tech/documentation/]",
      "mem0": "N/A (not publicly available) [source: https://docs.mem0.ai/]",
      "Zep": "N/A (not publicly available) [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Metadata filtering",
      "SQLite + embeddings": "Category + priority handled in schema [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "N/A (not publicly available) [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Yes [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Yes [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "Yes [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "N/A (not publicly available) [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Yes [source: https://qdrant.tech/documentation/]",
      "mem0": "N/A (not publicly available) [source: https://docs.mem0.ai/]",
      "Zep": "Yes [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Auto-expiry / TTL",
      "SQLite + embeddings": "Yes for short-term tier [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "N/A (not publicly available) [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "App-defined [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "N/A (not publicly available) [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "N/A (not publicly available) [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "Yes [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "App-defined [source: https://qdrant.tech/documentation/]",
      "mem0": "N/A (not publicly available) [source: https://docs.mem0.ai/]",
      "Zep": "N/A (not publicly available) [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Deduplication support",
      "SQLite + embeddings": "Yes, similarity + category-aware updates [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "No [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "N/A (not publicly available) [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "N/A (not publicly available) [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "N/A (not publicly available) [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "Manual overwrite only [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "N/A (not publicly available) [source: https://qdrant.tech/documentation/]",
      "mem0": "Yes, memory-layer style consolidation [source: https://docs.mem0.ai/]",
      "Zep": "N/A (not publicly available) [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Startup preload support",
      "SQLite + embeddings": "Yes, priority facts injected before session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "No [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Possible app-side [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "Possible app-side [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "N/A (not publicly available) [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "N/A (not publicly available) [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "N/A (not publicly available) [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "N/A (not publicly available) [source: https://qdrant.tech/documentation/]",
      "mem0": "N/A (not publicly available) [source: https://docs.mem0.ai/]",
      "Zep": "N/A (not publicly available) [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "Possible app-side [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Monthly infra cost",
      "SQLite + embeddings": "$0 extra service cost [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "$0 memory infra but no persistence [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "N/A (not publicly available) [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "External service cost [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "External service cost [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "External service cost [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "Higher ops cost [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Server/self-host cost [source: https://qdrant.tech/documentation/]",
      "mem0": "Service/framework cost [source: https://docs.mem0.ai/]",
      "Zep": "Service cost [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "N/A (not publicly available) [source: https://langchain-ai.github.io/langgraph/]"
    },
    {
      "Metric": "Offline / portable",
      "SQLite + embeddings": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py]",
      "In-context only": "Session only [source: https://platform.openai.com/docs/guides/prompt-caching]",
      "Chroma": "Can be local [source: https://docs.trychroma.com/docs/overview/introduction]",
      "Pinecone serverless": "No [source: https://docs.pinecone.io/docs/overview]",
      "Weaviate Cloud": "No [source: https://weaviate.io/developers/weaviate]",
      "Redis key-value": "No, external service/process [source: https://redis.io/docs/latest/]",
      "Full RAG (LangChain + FAISS)": "Partial [source: https://python.langchain.com/docs/introduction/]",
      "Qdrant": "Yes if self-hosted [source: https://qdrant.tech/documentation/]",
      "mem0": "No [source: https://docs.mem0.ai/]",
      "Zep": "No [source: https://help.getzep.com/]",
      "LangGraph checkpointing": "Partial [source: https://langchain-ai.github.io/langgraph/]"
    }
  ],
  "criteria": [
    "persistence",
    "low ops",
    "semantic recall",
    "startup preload"
  ],
  "fit_matrix": {
    "SQLite + embeddings": [
      1,
      1,
      1,
      1
    ],
    "In-context only": [
      0,
      1,
      0,
      0
    ],
    "Chroma": [
      1,
      0.4,
      1,
      0.5
    ],
    "Pinecone serverless": [
      1,
      0,
      1,
      0.5
    ],
    "Weaviate Cloud": [
      1,
      0,
      1,
      0.5
    ],
    "Redis key-value": [
      1,
      0.3,
      0,
      0.5
    ],
    "Full RAG (LangChain + FAISS)": [
      1,
      0,
      1,
      0.5
    ],
    "Qdrant": [
      1,
      0.3,
      1,
      0.5
    ],
    "mem0": [
      1,
      0.3,
      1,
      0.7
    ],
    "Zep": [
      1,
      0.2,
      1,
      0.6
    ],
    "LangGraph checkpointing": [
      1,
      0.4,
      0.5,
      0.5
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
labels=list(data['fit_matrix'].keys())
values=[sum(v) for v in data['fit_matrix'].values()]
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 5 - Memory Architecture Comparison Overall Fit Score")
ax.bar(labels,values,color=colors); ax.set_ylabel('Derived fit score',color='white'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(charts_dir / 'category5_memory_architecture_comparison_chart1.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["SQLite + embeddings", "mem0", "Qdrant", "Chroma", "Pinecone serverless"]; x=np.arange(len(criteria)); width=0.8/len(selected)
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 5 - Memory Architecture Comparison Top-5 Criteria View")
for idx,label in enumerate(selected):
 vals=data['fit_matrix'][label]; color=ALLY if label==selected[0] else (DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else COMP); ax.bar(x+(idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white'); plt.tight_layout(); plt.savefig(charts_dir / 'category5_memory_architecture_comparison_chart2.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
    colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label == list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
    ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["SQLite + embeddings", "mem0", "Qdrant", "Chroma", "Pinecone serverless"]; mat=np.array([data['fit_matrix'][k] for k in selected])
fig,ax=plt.subplots(figsize=(10,5)); ax.set_facecolor(BG); ax.figure.set_facecolor(BG); im=ax.imshow(mat,cmap='Blues',aspect='auto'); ax.set_title('Ally Vision v2 — Category 5 - Memory Architecture Comparison Trade-off Heatmap',color='white'); ax.set_xticks(np.arange(len(criteria))); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.set_yticks(np.arange(len(selected))); ax.set_yticklabels(selected,color='white')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]): ax.text(j,i,f"{mat[i,j]:.0f}",ha='center',va='center',color='white')
plt.colorbar(im); plt.tight_layout(); plt.savefig(charts_dir / 'category5_memory_architecture_comparison_chart3.png',dpi=150,bbox_inches='tight'); plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Memory store code | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_store.py | current SQLite schema and recall logic |
| 2 | Memory manager code | https://github.com/omshivarjun27/Blind-Assistance/blob/main/core/memory/memory_manager.py | startup preload and explicit save/recall behavior |
| 3 | Realtime route | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py | priority preload into session instructions |
| 4 | Python sqlite3 docs | https://docs.python.org/3/library/sqlite3.html | SQLite portability and zero-extra-service baseline |


## CONCLUSION

SQLite plus embeddings remains the strongest fit for Ally Vision v2 because it delivers persistent semantic recall, startup preload, and category-aware updates without introducing another always-on service. For a local-first assistive application, that combination is more valuable than adopting a heavier hosted vector stack by default.

→ Chosen for Ally Vision v2 ✅
